In [2]:
from pathlib import Path

In [3]:
import numpy as np
import polars as pl

In [6]:
# Imports
import os, math
import json
from pathlib import Path

## 2. Load & Merge Source Files

In [7]:
base_path = Path("../data/")

In [8]:
# Input file paths
INPUT_FILE1 = base_path / "CrowdCharge_Transactions.parquet"
INPUT_FILE2 = base_path / "GreenFlux_Transactions.parquet"

In [9]:
# Load parquet files and align schemas before merging
df1 = pl.read_parquet(INPUT_FILE1)[:, :21]
df2 = pl.read_parquet(INPUT_FILE2)[:, :21]

# Rename df2 columns to match df1 column order
df2 = df2.rename({old: new for old, new in zip(df2.columns, df1.columns)})

# Align dtypes between datasets
df1 = df1.with_columns(
    pl.col("ChargerID").cast(pl.Utf8),
    pl.col("Max_Current_Drawn_for_T").cast(pl.Float64),
)

# Concatenate and save merged file
df = pl.concat([df1, df2])
df.write_parquet("merged.parquet")

print(f"Merged dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Unique chargers: {df.select(pl.col('ChargerID').n_unique()).item():,}")
df.head(8)

Merged dataset: 157,520 rows, 21 columns
Unique chargers: 602


TransactionID,ChargerID,ParticipantID,CarKW,CarKWh,StartTime,StopTime,GroupID,Trial,AdjustedStartTime,AdjustedStopTime,PluggedInTime,ConsumedkWh,Part_of_Managed_Group,Weekday_or_Weekend,ActiveCharging_Start,Max_Current_Drawn_for_T,EndCharge,ChargingDuration,t_inactive_start,t_inactive_end
i64,str,str,f64,f64,datetime[ns],datetime[ns],str,f64,datetime[ns],datetime[ns],i64,f64,str,str,datetime[ns],f64,datetime[ns],f64,f64,f64
100000,"""35000117""","""EN1306""",7.0,33.0,2017-05-04 19:02:00,2017-05-05 08:47:00,null,null,2017-05-04 20:02:00,2017-05-05 09:47:00,825,29.39,"""No""","""Weekday""",null,null,null,null,null,null
100001,"""35000117""","""EN1306""",7.0,33.0,2017-05-05 17:15:00,2017-05-06 09:53:00,null,null,2017-05-05 18:15:00,2017-05-06 10:53:00,998,19.35,"""No""","""Weekday""",null,null,null,null,null,null
100002,"""35000117""","""EN1306""",7.0,33.0,2017-05-11 16:44:00,2017-05-12 06:18:00,null,null,2017-05-11 17:44:00,2017-05-12 07:18:00,814,19.17,"""No""","""Weekday""",null,null,null,null,null,null
100003,"""35000117""","""EN1306""",7.0,33.0,2017-05-15 18:47:00,2017-05-16 06:13:00,null,null,2017-05-15 19:47:00,2017-05-16 07:13:00,686,22.07,"""No""","""Weekday""",null,null,null,null,null,null
100004,"""35000117""","""EN1306""",7.0,33.0,2017-05-17 15:32:00,2017-05-18 06:44:00,null,null,2017-05-17 16:32:00,2017-05-18 07:44:00,912,27.47,"""No""","""Weekday""",null,null,null,null,null,null
100005,"""35000117""","""EN1306""",7.0,33.0,2017-05-22 11:37:00,2017-05-22 15:33:00,null,null,2017-05-22 12:37:00,2017-05-22 16:33:00,236,28.64,"""No""","""Weekday""",null,null,null,null,null,null
100006,"""35000117""","""EN1306""",7.0,33.0,2017-05-25 17:58:00,2017-06-01 12:51:00,null,null,2017-05-25 18:58:00,2017-06-01 13:51:00,9773,58.49,"""No""","""Weekday""",null,null,null,null,null,null
100007,"""35000117""","""EN1306""",7.0,33.0,2017-06-04 12:22:00,2017-06-04 16:22:00,null,null,2017-06-04 13:22:00,2017-06-04 17:22:00,240,18.35,"""No""","""Weekend""",null,null,null,null,null,null


## 3. Filter Outliers & Select Columns

In [10]:
# Remove outlier sessions (PluggedInTime outside 10–10,000 minutes)
df = df.filter(
    (pl.col("PluggedInTime") >= 10) &
    (pl.col("PluggedInTime") <= 10000)
)

print(f"After outlier filter: {df.shape[0]:,} rows")
df.head()

After outlier filter: 154,302 rows


TransactionID,ChargerID,ParticipantID,CarKW,CarKWh,StartTime,StopTime,GroupID,Trial,AdjustedStartTime,AdjustedStopTime,PluggedInTime,ConsumedkWh,Part_of_Managed_Group,Weekday_or_Weekend,ActiveCharging_Start,Max_Current_Drawn_for_T,EndCharge,ChargingDuration,t_inactive_start,t_inactive_end
i64,str,str,f64,f64,datetime[ns],datetime[ns],str,f64,datetime[ns],datetime[ns],i64,f64,str,str,datetime[ns],f64,datetime[ns],f64,f64,f64
100000,"""35000117""","""EN1306""",7.0,33.0,2017-05-04 19:02:00,2017-05-05 08:47:00,null,null,2017-05-04 20:02:00,2017-05-05 09:47:00,825,29.39,"""No""","""Weekday""",null,null,null,null,null,null
100001,"""35000117""","""EN1306""",7.0,33.0,2017-05-05 17:15:00,2017-05-06 09:53:00,null,null,2017-05-05 18:15:00,2017-05-06 10:53:00,998,19.35,"""No""","""Weekday""",null,null,null,null,null,null
100002,"""35000117""","""EN1306""",7.0,33.0,2017-05-11 16:44:00,2017-05-12 06:18:00,null,null,2017-05-11 17:44:00,2017-05-12 07:18:00,814,19.17,"""No""","""Weekday""",null,null,null,null,null,null
100003,"""35000117""","""EN1306""",7.0,33.0,2017-05-15 18:47:00,2017-05-16 06:13:00,null,null,2017-05-15 19:47:00,2017-05-16 07:13:00,686,22.07,"""No""","""Weekday""",null,null,null,null,null,null
100004,"""35000117""","""EN1306""",7.0,33.0,2017-05-17 15:32:00,2017-05-18 06:44:00,null,null,2017-05-17 16:32:00,2017-05-18 07:44:00,912,27.47,"""No""","""Weekday""",null,null,null,null,null,null


In [11]:
# Select relevant columns
df = df.select([
    "ChargerID",
    "AdjustedStartTime",
    "PluggedInTime",
    "ActiveCharging_Start",
    "ChargingDuration",
    "ConsumedkWh",
    "AdjustedStopTime",
    "Part_of_Managed_Group",
])

# Drop sessions with no recorded charging start
df = df.filter(pl.col("ActiveCharging_Start").is_not_null())

print(f"After column selection and null filter: {df.shape[0]:,} rows")
df.head(8)

After column selection and null filter: 111,179 rows


ChargerID,AdjustedStartTime,PluggedInTime,ActiveCharging_Start,ChargingDuration,ConsumedkWh,AdjustedStopTime,Part_of_Managed_Group
str,datetime[ns],i64,datetime[ns],f64,f64,datetime[ns],str
"""35000117""",2017-06-28 21:04:00,601,2017-06-28 21:05:00,281.0,31.23,2017-06-29 07:05:00,"""No"""
"""35000117""",2017-07-03 18:17:00,864,2017-07-03 18:19:00,233.0,26.24,2017-07-04 08:41:00,"""No"""
"""35000117""",2017-07-07 15:28:00,1420,2017-07-07 15:29:00,187.0,20.92,2017-07-08 15:08:00,"""No"""
"""35000117""",2017-07-13 22:35:00,5026,2017-07-13 22:36:00,264.0,30.59,2017-07-17 10:21:00,"""No"""
"""35000117""",2017-08-03 21:34:00,732,2017-08-03 21:35:00,220.0,24.88,2017-08-04 09:46:00,"""No"""
"""35000117""",2017-08-11 15:56:00,2670,2017-08-11 15:57:00,268.0,30.71,2017-08-13 12:26:00,"""No"""
"""35000117""",2017-08-18 17:24:00,921,2017-08-18 17:25:00,265.0,30.0,2017-08-19 08:45:00,"""No"""
"""35000117""",2018-01-15 17:25:00,858,2018-01-15 17:26:00,249.0,32.1,2018-01-16 07:43:00,"""No"""


## 4. Feature Engineering

In [12]:
# Ensure datetime columns are correctly typed
NS_PER_HOUR = 3_600_000_000_000

for col in ["ActiveCharging_Start", "AdjustedStopTime"]:
    if df.schema[col] != pl.Datetime:
        df = df.with_columns(
            pl.col(col)
              .cast(pl.Utf8)
              .str.strip_chars()
              .str.replace_all("  +", " ")
              .str.to_datetime(strict=False)
        )

# Compute ChargingDuration (hours) where missing
if "ChargingDuration" not in df.columns:
    df = df.with_columns(pl.lit(None, dtype=pl.Duration).alias("ChargingDuration"))

df = df.with_columns(
    pl.when(
        pl.col("ChargingDuration").is_null()
        & pl.col("ActiveCharging_Start").is_not_null()
        & pl.col("AdjustedStopTime").is_not_null()
    )
    .then(
        (pl.col("AdjustedStopTime").dt.epoch("ns") - pl.col("ActiveCharging_Start").dt.epoch("ns"))
        / NS_PER_HOUR
    )
    .otherwise(pl.col("ChargingDuration"))
    .cast(pl.Float64)
    .alias("ChargingDuration")
)

df.head()

ChargerID,AdjustedStartTime,PluggedInTime,ActiveCharging_Start,ChargingDuration,ConsumedkWh,AdjustedStopTime,Part_of_Managed_Group
str,datetime[ns],i64,datetime[ns],f64,f64,datetime[ns],str
"""35000117""",2017-06-28 21:04:00,601,2017-06-28 21:05:00,281.0,31.23,2017-06-29 07:05:00,"""No"""
"""35000117""",2017-07-03 18:17:00,864,2017-07-03 18:19:00,233.0,26.24,2017-07-04 08:41:00,"""No"""
"""35000117""",2017-07-07 15:28:00,1420,2017-07-07 15:29:00,187.0,20.92,2017-07-08 15:08:00,"""No"""
"""35000117""",2017-07-13 22:35:00,5026,2017-07-13 22:36:00,264.0,30.59,2017-07-17 10:21:00,"""No"""
"""35000117""",2017-08-03 21:34:00,732,2017-08-03 21:35:00,220.0,24.88,2017-08-04 09:46:00,"""No"""


In [13]:
# Encode Part_of_Managed_Group as binary integer
df = df.with_columns(
    pl.col("Part_of_Managed_Group")
      .replace({"Yes": 1, "No": 0})
      .cast(pl.Int8)
      .alias("Part_of_Managed_Group")
)

# Convert PluggedInTime and ChargingDuration from minutes to hours
df = df.with_columns(
    pl.col("PluggedInTime").cast(pl.Float64) / 60,
    pl.col("ChargingDuration").cast(pl.Float64) / 60,
)

# Compute time gap between plug-in and charging start (hours)
df = df.with_columns(
    (
        (pl.col("ActiveCharging_Start").dt.epoch("ns") - pl.col("AdjustedStartTime").dt.epoch("ns"))
        / 3_600_000_000_000
    ).cast(pl.Float64).clip(lower_bound=0).alias("TimeGapHours"),

    # Temporal features
    pl.col("AdjustedStartTime").dt.hour().cast(pl.Float64).alias("HourOfDay"),
    pl.col("AdjustedStartTime").dt.weekday().cast(pl.Float64).alias("DayOfWeek"),
)

# Shift DayOfWeek so Monday = 0
df = df.with_columns(
    (pl.col("DayOfWeek") - 1).alias("DayOfWeek")
)

df.head()

ChargerID,AdjustedStartTime,PluggedInTime,ActiveCharging_Start,ChargingDuration,ConsumedkWh,AdjustedStopTime,Part_of_Managed_Group,TimeGapHours,HourOfDay,DayOfWeek
str,datetime[ns],f64,datetime[ns],f64,f64,datetime[ns],i8,f64,f64,f64
"""35000117""",2017-06-28 21:04:00,10.016667,2017-06-28 21:05:00,4.683333,31.23,2017-06-29 07:05:00,0,0.016667,21.0,2.0
"""35000117""",2017-07-03 18:17:00,14.4,2017-07-03 18:19:00,3.883333,26.24,2017-07-04 08:41:00,0,0.033333,18.0,0.0
"""35000117""",2017-07-07 15:28:00,23.666667,2017-07-07 15:29:00,3.116667,20.92,2017-07-08 15:08:00,0,0.016667,15.0,4.0
"""35000117""",2017-07-13 22:35:00,83.766667,2017-07-13 22:36:00,4.4,30.59,2017-07-17 10:21:00,0,0.016667,22.0,3.0
"""35000117""",2017-08-03 21:34:00,12.2,2017-08-03 21:35:00,3.666667,24.88,2017-08-04 09:46:00,0,0.016667,21.0,3.0


In [14]:
# Cyclic encoding of session start time within the week
SECONDS_PER_DAY  = 86_400
SECONDS_PER_WEEK = 604_800

week_position = (
    (pl.col("AdjustedStartTime").dt.weekday() - 1).cast(pl.Int64) * SECONDS_PER_DAY
    + (pl.col("AdjustedStartTime").dt.epoch("s") % SECONDS_PER_DAY)
)

df = df.with_columns([
    ((2 * math.pi) * week_position / SECONDS_PER_WEEK).sin().alias("WeekTime_sin"),
    ((2 * math.pi) * week_position / SECONDS_PER_WEEK).cos().alias("WeekTime_cos"),
])

print(f"Final dataset: {df.shape[0]:,} rows")
df.head()

Final dataset: 111,179 rows


ChargerID,AdjustedStartTime,PluggedInTime,ActiveCharging_Start,ChargingDuration,ConsumedkWh,AdjustedStopTime,Part_of_Managed_Group,TimeGapHours,HourOfDay,DayOfWeek,WeekTime_sin,WeekTime_cos
str,datetime[ns],f64,datetime[ns],f64,f64,datetime[ns],i8,f64,f64,f64,f64,f64
"""35000117""",2017-06-28 21:04:00,10.016667,2017-06-28 21:05:00,4.683333,31.23,2017-06-29 07:05:00,0,0.016667,21.0,2.0,0.529919,-0.848048
"""35000117""",2017-07-03 18:17:00,14.4,2017-07-03 18:19:00,3.883333,26.24,2017-07-04 08:41:00,0,0.033333,18.0,0.0,0.631739,0.775181
"""35000117""",2017-07-07 15:28:00,23.666667,2017-07-07 15:29:00,3.116667,20.92,2017-07-08 15:08:00,0,0.016667,15.0,4.0,-0.85588,-0.517174
"""35000117""",2017-07-13 22:35:00,83.766667,2017-07-13 22:36:00,4.4,30.59,2017-07-17 10:21:00,0,0.016667,22.0,3.0,-0.385561,-0.922682
"""35000117""",2017-08-03 21:34:00,12.2,2017-08-03 21:35:00,3.666667,24.88,2017-08-04 09:46:00,0,0.016667,21.0,3.0,-0.350207,-0.936672


## 5. Validation

In [15]:
# Summary statistics for a mid-week slice (Wednesday, DayOfWeek == 2)
wednesday = df.filter(pl.col("DayOfWeek") == 2).select([
    "PluggedInTime", "ChargingDuration", "ConsumedkWh",
    "TimeGapHours", "WeekTime_sin", "WeekTime_cos"
])

print(f"Wednesday session count: {wednesday.shape[0]:,}")
print("\nMean:\n",  wednesday.select(pl.all().mean()))
print("\nStd:\n",   wednesday.select(pl.all().std()))

Wednesday session count: 16,389

Mean:
 shape: (1, 6)
┌───────────────┬──────────────────┬─────────────┬──────────────┬──────────────┬──────────────┐
│ PluggedInTime ┆ ChargingDuration ┆ ConsumedkWh ┆ TimeGapHours ┆ WeekTime_sin ┆ WeekTime_cos │
│ ---           ┆ ---              ┆ ---         ┆ ---          ┆ ---          ┆ ---          │
│ f64           ┆ f64              ┆ f64         ┆ f64          ┆ f64          ┆ f64          │
╞═══════════════╪══════════════════╪═════════════╪══════════════╪══════════════╪══════════════╡
│ 11.522936     ┆ 2.35149          ┆ 10.651873   ┆ 1.223974     ┆ 0.63241      ┆ -0.760117    │
└───────────────┴──────────────────┴─────────────┴──────────────┴──────────────┴──────────────┘

Std:
 shape: (1, 6)
┌───────────────┬──────────────────┬─────────────┬──────────────┬──────────────┬──────────────┐
│ PluggedInTime ┆ ChargingDuration ┆ ConsumedkWh ┆ TimeGapHours ┆ WeekTime_sin ┆ WeekTime_cos │
│ ---           ┆ ---              ┆ ---         ┆ ---       

## 6. Save Outputs

In [16]:
# Save full cleaned dataset (with timestamps retained for reference)
FULL_CLEANED_PATH = base_path / "processed" / "Real_transactions_cleaned.parquet"
df.write_parquet(FULL_CLEANED_PATH)
print(f"Saved full cleaned data: {df.shape[0]:,} rows → {FULL_CLEANED_PATH}")

Saved full cleaned data: 111,179 rows → ../data/processed/Real_transactions_cleaned.parquet


In [17]:
# Drop columns not needed for model training and save pre-scale dataset
df_model = df.drop(
    "ChargerID", "AdjustedStartTime", "ActiveCharging_Start",
    "AdjustedStopTime", "HourOfDay"
)

PRE_SCALE = base_path / "processed" / "Transactions_preprocessedV3.4.7_combined.parquet"
df_model.write_parquet(PRE_SCALE)
print(f"Saved pre-scale model input: {df_model.shape[0]:,} rows → {PRE_SCALE}")
df_model.head()

Saved pre-scale model input: 111,179 rows → ../data/processed/Transactions_preprocessedV3.4.7_combined.parquet


PluggedInTime,ChargingDuration,ConsumedkWh,Part_of_Managed_Group,TimeGapHours,DayOfWeek,WeekTime_sin,WeekTime_cos
f64,f64,f64,i8,f64,f64,f64,f64
10.016667,4.683333,31.23,0,0.016667,2.0,0.529919,-0.848048
14.4,3.883333,26.24,0,0.033333,0.0,0.631739,0.775181
23.666667,3.116667,20.92,0,0.016667,4.0,-0.85588,-0.517174
83.766667,4.4,30.59,0,0.016667,3.0,-0.385561,-0.922682
12.2,3.666667,24.88,0,0.016667,3.0,-0.350207,-0.936672
